# 03 — Exploratory Data Analysis (EDA)

**Objective:** Understand the purchasing behaviour represented by the cleaned dataset through statistical summaries and visualizations.

### EDA Deliverables
1. Top products chart
2. Monthly sales chart
3. Country-wise transaction chart
4. Basket-size distribution
5. Statistical summary

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 2. Load the Cleaned Dataset

In [ ]:
df = pd.read_csv('../dataset/processed/cleaned_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Cleaned dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

---
## 3. Statistical Summary

Calculate key metrics about the transactions and basket composition.

In [ ]:
# Basket = number of unique items per invoice
basket_sizes = df.groupby('InvoiceNo')['Description'].nunique()

# Quantity per transaction
qty_per_txn = df.groupby('InvoiceNo')['Quantity'].sum()

total_transactions = df['InvoiceNo'].nunique()
avg_basket_size    = round(basket_sizes.mean(), 2)
max_basket_size    = basket_sizes.max()
min_basket_size    = basket_sizes.min()
unique_products    = df['Description'].nunique()
avg_qty_per_txn    = round(qty_per_txn.mean(), 2)

print("=" * 50)
print("         STATISTICAL SUMMARY")
print("=" * 50)
print(f"  Total Transactions             : {total_transactions:>10,}")
print(f"  Average Basket Size            : {avg_basket_size:>10}")
print(f"  Maximum Basket Size            : {max_basket_size:>10}")
print(f"  Minimum Basket Size            : {min_basket_size:>10}")
print(f"  Unique Products                : {unique_products:>10,}")
print(f"  Avg Quantity per Transaction   : {avg_qty_per_txn:>10}")
print("=" * 50)

**Observations — Statistical Summary:**
- The dataset contains **35,338 unique transactions** across **3,988 unique products**.
- On average, each transaction (basket) contains about **21 unique items**, indicating customers tend to buy many different products per visit.
- The largest basket has **541 items** — likely a bulk or wholesale order.
- Some baskets contain only **1 item**, representing single-item purchases.
- Average total quantity per transaction is approximately **251 units**, suggesting many items are bought in bulk.

---
## 4. Top 10 Most Frequently Purchased Items (by Total Quantity)

Identify the products with the highest total quantity sold across all transactions.

In [ ]:
top10_qty = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=top10_qty.values, y=top10_qty.index, palette='viridis')
plt.title('Top 10 Most Frequently Purchased Items (by Total Quantity)', fontsize=14, fontweight='bold')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Product Description')

# Add value labels on bars
for i, v in enumerate(top10_qty.values):
    ax.text(v + 500, i, f'{v:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/graphs/top10_items_by_quantity.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved to: ../results/graphs/top10_items_by_quantity.png")

**Observations — Top 10 Items by Quantity:**
- **PAPER CRAFT, LITTLE BIRDIE** dominates with over 161,000 units sold — nearly double the second-highest item.
- Items like **MEDIUM CERAMIC TOP STORAGE JAR** and **WORLD WAR 2 GLIDERS** also show very high volumes, suggesting they are either popular low-cost items or frequently bought in bulk.
- **WHITE HANGING HEART T-LIGHT HOLDER** and **JUMBO BAG RED RETROSPOT** appear in this list, confirming they are high-volume products.
- These products are strong candidates for placement in association rules as frequent antecedents.

---
## 5. Top 10 Most Frequently Purchased Products (by Invoice Count)

Identify products that appear in the most number of unique transactions (invoices), regardless of quantity.

In [ ]:
top10_inv = df.groupby('Description')['InvoiceNo'].nunique().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=top10_inv.values, y=top10_inv.index, palette='magma')
plt.title('Top 10 Products by Invoice Count (Unique Transactions)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Unique Invoices')
plt.ylabel('Product Description')

# Add value labels on bars
for i, v in enumerate(top10_inv.values):
    ax.text(v + 20, i, f'{v:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/graphs/top10_items_by_invoice_count.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved to: ../results/graphs/top10_items_by_invoice_count.png")

**Observations — Top 10 Products by Invoice Count:**
- **WHITE HANGING HEART T-LIGHT HOLDER** appears in the most invoices (~2,969), making it the most popular product across distinct transactions.
- **REGENCY CAKESTAND 3 TIER** and **JUMBO BAG RED RETROSPOT** are also purchased frequently across many transactions.
- Note: **POSTAGE** appears in this list (2,080 invoices) — this is a service charge, not a product. It may need special handling during association rule mining.
- Products that appear in many invoices have **high support values** and are more likely to form strong association rules.
- The difference between this chart and the quantity chart highlights that high-quantity items are not always the ones bought across the most transactions — some are bought in bulk in fewer orders.

---
## 6. Monthly Sales Analysis

Analyze revenue trends over time to understand seasonality and growth patterns.

In [ ]:
# Calculate total revenue per month
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

monthly_sales = df.groupby('YearMonth')['TotalPrice'].sum()

plt.figure(figsize=(14, 6))
plt.plot(monthly_sales.index.astype(str), monthly_sales.values, 
         marker='o', linewidth=2, markersize=6, color='#2196F3')
plt.fill_between(range(len(monthly_sales)), monthly_sales.values, alpha=0.15, color='#2196F3')
plt.title('Monthly Sales Revenue Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Revenue (£)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/graphs/monthly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved to: ../results/graphs/monthly_sales_trend.png")

In [ ]:
# Monthly sales table
monthly_df = monthly_sales.reset_index()
monthly_df.columns = ['Month', 'Revenue (£)']
monthly_df['Revenue (£)'] = monthly_df['Revenue (£)'].round(2)
monthly_df

**Observations — Monthly Sales:**
- There is a clear **seasonal pattern**: sales rise sharply in **September–November** each year, peaking in November — likely driven by holiday and Christmas shopping.
- **November 2012** recorded the highest revenue at £1,438,244, followed by **November 2011** at £1,156,205.
- **February** consistently shows the lowest revenue, suggesting a post-holiday dip in purchasing.
- **December 2012** shows a sharp drop because the data only covers part of the month (up to December 9).
- Overall, revenue shows an **upward trend** from 2011 to 2012, indicating business growth.

---
## 7. Country-wise Transactions

Analyze the geographical distribution of transactions.

In [ ]:
country_txn = df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False)

# Plot top 10 countries (excluding UK for better scale of other countries)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: All top 10 including UK
top10_countries = country_txn.head(10)
ax1 = sns.barplot(x=top10_countries.values, y=top10_countries.index, palette='coolwarm', ax=axes[0])
axes[0].set_title('Top 10 Countries by Transaction Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Transactions')
axes[0].set_ylabel('Country')
for i, v in enumerate(top10_countries.values):
    ax1.text(v + 100, i, f'{v:,}', va='center', fontsize=9)

# Right: Top 10 countries EXCLUDING UK (to see the detail)
top10_no_uk = country_txn[country_txn.index != 'United Kingdom'].head(10)
ax2 = sns.barplot(x=top10_no_uk.values, y=top10_no_uk.index, palette='coolwarm', ax=axes[1])
axes[1].set_title('Top 10 Countries (Excluding UK)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Transactions')
axes[1].set_ylabel('')
for i, v in enumerate(top10_no_uk.values):
    ax2.text(v + 5, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/graphs/country_wise_transactions.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved to: ../results/graphs/country_wise_transactions.png")

In [ ]:
# UK dominance percentage
uk_pct = round(country_txn['United Kingdom'] / country_txn.sum() * 100, 2)
print(f"United Kingdom accounts for {uk_pct}% of all transactions")
print(f"\nTotal countries: {len(country_txn)}")

**Observations — Country-wise Transactions:**
- **United Kingdom** overwhelmingly dominates with **31,764 transactions** (~89.9% of all transactions).
- **Germany** (925), **France** (700), and **EIRE/Ireland** (386) are the next biggest markets but are significantly smaller.
- The right chart (excluding UK) reveals that European countries form the bulk of international orders.
- For market basket analysis, it may be beneficial to **focus on UK transactions** to avoid country-specific purchasing patterns skewing the association rules.
- Alternatively, country can be used as a filter to compare cross-selling patterns across markets.

---
## 8. Distribution of Basket Sizes

Analyze how many unique items customers typically purchase per transaction.

In [ ]:
basket_sizes = df.groupby('InvoiceNo')['Description'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Full distribution
axes[0].hist(basket_sizes, bins=50, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Basket Sizes (All)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Unique Items per Transaction')
axes[0].set_ylabel('Frequency')
axes[0].axvline(basket_sizes.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {basket_sizes.mean():.1f}')
axes[0].axvline(basket_sizes.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median = {basket_sizes.median():.0f}')
axes[0].legend()

# Right: Zoomed in (basket size <= 80 for detail)
basket_zoomed = basket_sizes[basket_sizes <= 80]
axes[1].hist(basket_zoomed, bins=80, color='#FF9800', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribution of Basket Sizes (Zoomed: ≤ 80 items)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Unique Items per Transaction')
axes[1].set_ylabel('Frequency')
axes[1].axvline(basket_sizes.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {basket_sizes.mean():.1f}')
axes[1].axvline(basket_sizes.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median = {basket_sizes.median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/graphs/basket_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved to: ../results/graphs/basket_size_distribution.png")

In [ ]:
# Basket size statistics
print("Basket Size Statistics:")
print("-" * 30)
print(basket_sizes.describe())

**Observations — Basket Size Distribution:**
- The distribution is **heavily right-skewed** — most transactions contain a small number of unique items, while a few contain hundreds.
- **Median basket size is 15 items**, while the mean is ~21, indicating the right tail of large baskets pulls the average up.
- **75% of transactions** contain 27 or fewer unique items.
- The largest basket contains **541 unique items** — likely a wholesale/B2B order.
- For association rule mining, very large baskets can generate noise. It may be useful to set an upper limit or segment by basket size.
- The majority of baskets (1–30 items) represent typical retail customer behavior and are ideal for discovering meaningful item associations.

---
## 9. EDA Summary

| Metric | Value |
|---|---|
| Total Transactions | 35,338 |
| Unique Products | 3,988 |
| Average Basket Size | ~21 items |
| Median Basket Size | 15 items |
| Max Basket Size | 541 items |
| Avg Quantity per Transaction | ~251 units |
| Countries | 38 |
| UK Transaction Share | ~89.9% |
| Peak Sales Month | November |
| Most Popular Product (by invoices) | WHITE HANGING HEART T-LIGHT HOLDER |
| Most Sold Product (by quantity) | PAPER CRAFT, LITTLE BIRDIE |

### Key Takeaways for Market Basket Analysis:
1. **High basket diversity** — average of 21 unique items per basket creates rich opportunities for finding item associations.
2. **UK-centric data** — over 89% of transactions are from the UK, so association rules will primarily reflect UK purchasing behavior.
3. **Seasonal patterns** — Q4 (Sep–Nov) is the peak buying season; associations may differ by season.
4. **Service items** like POSTAGE appear frequently and should be considered for exclusion from association rules.
5. **Right-skewed basket sizes** — a small number of very large baskets exist that may need filtering.

---
*Next step → `04_apriori.ipynb` — Apply the Apriori algorithm to mine frequent itemsets.*